# Red neural: temperatura de equilibrio de planetas

Aquí, vamos a tomar un problema diferente, pero similar al anterior:

__En vez de clasificar, vamos a hacer una regresión__

Si en los ejemplos anteriores, la última capa la activábamos con una escalón o una sigmoide para expresar la certeza o probabilidad de que un dato fuera 0 o 1, aquí, vamos a utilizar una activación lineal para que el output total de la red sea cualquier número que le plazca.

Para ello, vamos a tomar el ejemplo de 

<p style="text-align: center;">Predecir la temperatura de equilibrio exoplanetas a partir de otros datos de ellos y su sistema solar</p>



## Conseguir los datos

Aquí, vamos a conseguir los datos del __Nasa Exoplanet Archive__, y vamos a filtrar las columnas:

*Nombre del planeta, Periodo orbital, Órbita Máxima, Radio del planeta, Insolación del planeta, Temperatura efectiva de la estrella, Tránsito del planeta, Radio estelar y Masa estelar*

y, al final, $\hat y$ será la *Temperatura de equilibrio del planeta*.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Leemos los datos
df = pd.read_csv("data.csv")

# Separamos inputs y target
X = df.drop(['pl_name', 'pl_eqt'], axis=1).values
y = df['pl_eqt'].values

# Creamos tests de entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

# Escalamos las feautres para que la red se alimente más fácil
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Creación de la NN

Vamos a usar __TensorFlow__ (por CPU) para entrenar la red neuronal. La red consistirá de 3 capas


$$
\begin{matrix}
inputs & 1ra\,capa & 2da\,capa & output \\
\mathbf{X} \quad \to  & 64 \textit{ neuronas} \to & 32 \textit{ neuronas} \to & 1 \textit{ neurona}\\
\textit{9 features} & \textit{Act. ReLu} & \textit{Act. ReLu} & \textit{Act. Lineal} 
\end{matrix}
$$

Utilizamos activación ReLu para tener no linealidad en las capas internas, además, como nuestros datos input no están acotados superiormente, ReLu ayuda a conservar estas cotas. En la última capa usamos activación lineal porque nuestro output es un número real sin cota. 

Los demás parámetros son el default para NNs.

In [66]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Creamos el modelo
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)   # la última capa tiene activación lineal porque queremos datos de 0 a infty
])

# Le decimos al modelo qué hacer
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

## Entrenamiento

Como la NN puede dar pasos equívocos, o converger mucho antes del límite que le habíamos puesto al principio, entonces agregamos un _early stop_ (a partir del set de validación), y en todo caso, guardamos también las mejores iteraciones del modelo como _checkpoints_.

Utilizamos un set de validación del 20%, y pasamos batches de 128 datos al CPU.

In [67]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Si en el decenso del gradiente se nos va por trochas que no son, entonces volvemos
# a donde estábamos bien
early_stop = EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True
)

# Guardamos checkpoints del modelo que mejor se comporan
checkpoint = ModelCheckpoint(
    'best_model.keras', monitor='val_loss', save_best_only=True
)

# Entrenamos
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,          
    epochs=500,
    batch_size=128,
    callbacks=[early_stop, checkpoint],
    verbose=0
)

## Evaluamos el modelo

Ahora, podemos evaluar el modelo en el set de testing. 

Elegimos la métrica MAE (Mean Average Error), pues esta nos dice más o menos por cuánto se descacha nuestro modelo al predecir temperaturas.

In [39]:
# Evaluamos el modelo en el test set
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test MAE: {test_mae:.2f}")   # Imprimios el Mean Average Error

#  Guardamos el modelo para luego
model.save('model.keras')   # or .h5 for older versions

Test MAE: 5.94


## Evaluamos manualmente

Ahora, elegimos un planeta cualquiera, y evaluamos cómo se comporta el modelo con él.

In [68]:
# Cargamos el modelo
loaded_model = tf.keras.models.load_model('model.keras')

# Función para hallar los datos de un planeta en específico
def get_data(pl_name, which = 0):

    # Hallamos los inputs
    row = df.loc[df["pl_name"] == pl_name]
    inputs = row.drop(['pl_name', 'pl_eqt'], axis=1).values[which]
    temp = row['pl_eqt'].values[which]

    print("Planetas posibles:")
    print(row)

    return inputs.reshape(1, -1), temp
    
# Cargamos los datos
new_data, new_temp = get_data("Kepler-328 c")
new_data_scaled = scaler.transform(new_data)

# Predecimos
print("\nPREDICIENDO ")
predictions = loaded_model.predict(new_data_scaled)

print("\nResultados ")
print(f"Temperatura reportada del modelo: {new_temp}")
print(f"Predicción del modelo: {predictions[0][0]}")
print(f"Error relativo: {(100.0*(1 - (new_temp/predictions[0][0]))):.2f}%")

Planetas posibles:
            pl_name  pl_orbper  pl_orbsmax  pl_rade  pl_insol  st_teff  \
10407  Kepler-328 c  71.311658      0.3526     5.18     11.79   6167.0   
10408  Kepler-328 c  71.311658      0.3526     5.18     11.79   6167.0   
10409  Kepler-328 c  71.311658      0.3526     5.18     11.79   6167.0   
10490  Kepler-328 c  71.311660      0.3540     4.94     12.01   6194.0   

       pl_eqt    pl_tranmid  st_rad  st_mass  
10407   473.0  2.454965e+06   1.064    1.149  
10408   473.0  2.454965e+06   1.064    1.149  
10409   473.0  2.454965e+06   1.064    1.149  
10490   475.0  2.454965e+06   1.069    1.161  

PREDICIENDO 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

Resultados 
Temperatura reportada del modelo: 473.0
Predicción del modelo: 473.4723815917969
Error relativo: 0.10%
